# ControlNet 条件控制生成教程

本教程讲解 ControlNet 的条件控制生成：
- 边缘检测控制 (Canny)
- 深度图控制 (Depth)
- 姿态控制 (Pose)
- 多条件组合

In [ ]:
import sys
sys.path.insert(0, '../src')

import torch
import torch.nn.functional as F
from typing import Dict, List, Optional

from controlnet import ControlNet, ControlNetConfig, create_controlnet

## 1. ControlNet 原理

### 零卷积机制

ControlNet 的核心创新是零卷积 (Zero Convolution)：

$$y = F(x) + \text{ZeroConv}(\text{ControlNet}(c))$$

- 初始时 ZeroConv 输出为 0，不影响原模型
- 训练过程中逐渐学习条件控制信号

In [ ]:
# 创建不同类型的 ControlNet
controlnet_canny = create_controlnet("canny", "tiny")
controlnet_depth = create_controlnet("depth", "tiny")
controlnet_pose = create_controlnet("pose", "tiny")

print(f"Canny ControlNet: {sum(p.numel() for p in controlnet_canny.parameters()):,} params")
print(f"Depth ControlNet: {sum(p.numel() for p in controlnet_depth.parameters()):,} params")

## 2. 边缘检测控制

In [ ]:
def simple_canny_edge(image: torch.Tensor, low_threshold: float = 0.1, high_threshold: float = 0.3) -> torch.Tensor:
    """
    简化的 Canny 边缘检测
    
    实际应用中使用 OpenCV 的 cv2.Canny()
    """
    # 转灰度
    if image.shape[1] == 3:
        gray = 0.299 * image[:, 0] + 0.587 * image[:, 1] + 0.114 * image[:, 2]
        gray = gray.unsqueeze(1)
    else:
        gray = image
    
    # Sobel 算子
    sobel_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]], dtype=torch.float32).view(1, 1, 3, 3)
    sobel_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]], dtype=torch.float32).view(1, 1, 3, 3)
    
    grad_x = F.conv2d(gray, sobel_x.to(gray.device), padding=1)
    grad_y = F.conv2d(gray, sobel_y.to(gray.device), padding=1)
    
    # 梯度幅值
    magnitude = torch.sqrt(grad_x ** 2 + grad_y ** 2)
    magnitude = magnitude / magnitude.max()
    
    # 阈值处理
    edges = (magnitude > low_threshold).float()
    
    return edges

# 测试边缘检测
test_image = torch.randn(1, 3, 256, 256)
edges = simple_canny_edge(test_image)
print(f"Edge map shape: {edges.shape}")

## 3. 深度图控制

In [ ]:
def simulate_depth_map(image: torch.Tensor) -> torch.Tensor:
    """
    模拟深度图 (实际使用 MiDaS 等深度估计模型)
    """
    # 简单模拟: 使用亮度作为深度
    if image.shape[1] == 3:
        depth = 0.299 * image[:, 0] + 0.587 * image[:, 1] + 0.114 * image[:, 2]
    else:
        depth = image[:, 0]
    
    # 归一化到 [0, 1]
    depth = (depth - depth.min()) / (depth.max() - depth.min() + 1e-8)
    
    return depth.unsqueeze(1)

depth_map = simulate_depth_map(test_image)
print(f"Depth map shape: {depth_map.shape}")

## 4. 条件强度控制

In [ ]:
class ControlNetPipeline:
    """
    ControlNet 生成管线
    """
    
    def __init__(self, controlnet: ControlNet):
        self.controlnet = controlnet
    
    def prepare_condition(self, condition_image: torch.Tensor, control_type: str) -> torch.Tensor:
        """准备条件图像"""
        if control_type == "canny":
            return simple_canny_edge(condition_image)
        elif control_type == "depth":
            return simulate_depth_map(condition_image)
        else:
            return condition_image
    
    def get_control_signals(self, latents: torch.Tensor, timesteps: torch.Tensor,
                           condition: torch.Tensor, context: torch.Tensor = None,
                           conditioning_scale: float = 1.0) -> Dict:
        """
        获取 ControlNet 控制信号
        
        Args:
            conditioning_scale: 控制强度 (0-2)
                - 0.0: 无控制
                - 1.0: 标准控制
                - >1.0: 增强控制
        """
        return self.controlnet(
            latents, timesteps, condition, context,
            conditioning_scale=conditioning_scale
        )

# 测试管线
pipeline = ControlNetPipeline(controlnet_canny)
latents = torch.randn(1, 4, 32, 32)
timesteps = torch.tensor([500])
condition = torch.randn(1, 1, 256, 256)  # Canny 边缘图

signals = pipeline.get_control_signals(latents, timesteps, condition, conditioning_scale=1.0)
print(f"Control signals: {len(signals['down_block_res_samples'])} down blocks")

## 5. 多条件组合

In [ ]:
class MultiControlNetPipeline:
    """
    多 ControlNet 组合管线
    
    支持同时使用多个条件控制
    """
    
    def __init__(self, controlnets: Dict[str, ControlNet]):
        self.controlnets = controlnets
    
    def get_combined_signals(self, latents: torch.Tensor, timesteps: torch.Tensor,
                            conditions: Dict[str, torch.Tensor],
                            scales: Dict[str, float] = None) -> Dict:
        """
        获取组合控制信号
        
        Args:
            conditions: {control_type: condition_image}
            scales: {control_type: scale}
        """
        scales = scales or {k: 1.0 for k in conditions}
        
        combined_down = None
        combined_mid = None
        
        for control_type, condition in conditions.items():
            if control_type not in self.controlnets:
                continue
            
            scale = scales.get(control_type, 1.0)
            signals = self.controlnets[control_type](
                latents, timesteps, condition, None, scale
            )
            
            if combined_down is None:
                combined_down = signals['down_block_res_samples']
                combined_mid = signals['mid_block_res_sample']
            else:
                # 累加控制信号
                combined_down = [a + b for a, b in zip(combined_down, signals['down_block_res_samples'])]
                combined_mid = combined_mid + signals['mid_block_res_sample']
        
        return {
            'down_block_res_samples': combined_down,
            'mid_block_res_sample': combined_mid
        }

# 测试多条件组合
multi_pipeline = MultiControlNetPipeline({
    'canny': controlnet_canny,
    'depth': controlnet_depth
})

conditions = {
    'canny': torch.randn(1, 1, 256, 256),
    'depth': torch.randn(1, 1, 256, 256)
}
scales = {'canny': 0.8, 'depth': 0.5}

combined = multi_pipeline.get_combined_signals(latents, timesteps, conditions, scales)
print(f"Combined control: {len(combined['down_block_res_samples'])} blocks")

## 总结

本教程介绍了 ControlNet 的核心技术：

1. **零卷积**: 训练初期不影响原模型
2. **边缘控制**: Canny 边缘检测引导生成
3. **深度控制**: 深度图控制空间结构
4. **多条件组合**: 同时使用多种控制信号